# SARIMAX Experiment

This notebook contains the external-regressor version of the SARIMA experiment. It keeps SARIMA disabled, but uses SARIMAX-style aggregate weekly exogenous variables from `features.csv`.

It tests:

1. small SARIMA order search;
2. SARIMAX with aggregate weekly external regressors;
3. simple feature selection for exogenous variables;
4. last-year vs blended allocation from weekly total forecast to Store-Dept rows.

Pure SARIMA without external regressors is kept in `model_experiment_SARIMA.ipynb`.


In [1]:
%pip install -q "numpy>=1.24,<3" "pandas>=2.0,<3" "scikit-learn>=1.3,<2" "statsmodels>=0.14,<1" "wandb>=0.19,<1" "cloudpickle>=3.0,<4"

from pathlib import Path
import itertools
import warnings
import cloudpickle

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    import wandb
except Exception:
    wandb = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

In [4]:
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

Mounted at /content/drive


## Configuration

The search is intentionally small. This file is the external-regressor SARIMAX counterpart; seasonal-order tuning can be handled separately if needed.

In [5]:
DATA_DIR_CANDIDATES = [
    Path("/content/drive/MyDrive/walmart_competition_data"),
    Path("/content/Walmart-Recruiting---Store-Sales-Forecasting/data"),
    Path("../../../../data"),
    Path("../../../data"),
    Path("data"),
]

VALIDATION_WEEKS = 39
HOLIDAY_WEIGHT = 5.0
ORDER_GRID = list(itertools.product([0, 1, 2], [0, 1], [0, 1, 2]))
USE_EXOG_OPTIONS = [True]
ALLOCATION_STRATEGIES = ["last_year_share", "blended_share"]
CORRELATION_THRESHOLD = 0.05
MAX_EXOG_FEATURES = 8

RUN_WANDB = True
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
WANDB_RUN_NAME = "SARIMAX_Order_Exog_Allocation_Experiment"
WANDB_MODE = "online"

RUN_TEST_SUBMISSION = False
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def resolve_data_dir(candidates):
    required = ["train.csv", "test.csv", "features.csv", "stores.csv"]
    for candidate in candidates:
        if all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError("Could not find Walmart data directory.")


def weighted_mae(y_true, y_pred, is_holiday, holiday_weight=5.0):
    weights = np.where(np.asarray(is_holiday).astype(bool), holiday_weight, 1.0)
    return float(np.sum(weights * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(weights))


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
print(f"Using data directory: {DATA_DIR.resolve()}")

Using data directory: /content/drive/MyDrive/walmart_competition_data


In [6]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
features = pd.read_csv(DATA_DIR / "features.csv", parse_dates=["Date"])
stores = pd.read_csv(DATA_DIR / "stores.csv")

train = train.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
test = test.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
features = features.sort_values(["Date", "Store"]).reset_index(drop=True)

print({
    "train": train.shape,
    "test": test.shape,
    "features": features.shape,
    "stores": stores.shape,
})
display(train.head())

{'train': (421570, 5), 'test': (115064, 4), 'features': (8190, 12), 'stores': (45, 3)}


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,2,2010-02-05,50605.27,False
2,1,3,2010-02-05,13740.12,False
3,1,4,2010-02-05,39954.04,False
4,1,5,2010-02-05,32229.38,False


In [7]:
all_dates = np.array(sorted(train["Date"].unique()))
validation_dates = all_dates[-VALIDATION_WEEKS:]
validation_start = validation_dates[0]

train_part = train[train["Date"] < validation_start].copy()
val_part = train[train["Date"] >= validation_start].copy()

print({
    "train_rows": len(train_part),
    "validation_rows": len(val_part),
    "train_end": train_part["Date"].max().date(),
    "validation_start": pd.Timestamp(validation_start).date(),
    "validation_end": val_part["Date"].max().date(),
})

{'train_rows': 305982, 'validation_rows': 115588, 'train_end': datetime.date(2012, 1, 27), 'validation_start': datetime.date(2012, 2, 3), 'validation_end': datetime.date(2012, 10, 26)}


## SARIMAX Feature Engineering and Feature Selection

This SARIMAX notebook uses aggregate weekly exogenous variables that are available for future dates from `features.csv`:

- holiday share;
- average temperature/fuel/CPI/unemployment;
- total markdown values;
- calendar month and week-of-year sin/cos.

Feature selection is simple and time-safe: it is fitted only on training weekly data, keeps variables correlated with weekly total sales, and removes highly collinear duplicates.

In [8]:
def weekly_total_sales(frame):
    return frame.groupby("Date", as_index=True)["Weekly_Sales"].sum().sort_index().asfreq("W-FRI")


def build_weekly_exog(features_frame):
    frame = features_frame.copy()
    markdown_cols = [c for c in frame.columns if c.startswith("MarkDown")]
    for col in markdown_cols:
        frame[col] = frame[col].fillna(0.0)

    numeric_cols = ["Temperature", "Fuel_Price", "CPI", "Unemployment"] + markdown_cols
    for col in numeric_cols:
        if col in frame.columns:
            frame[col] = frame[col].fillna(frame[col].median())

    aggregations = {
        "IsHoliday": "mean",
        "Temperature": "mean",
        "Fuel_Price": "mean",
        "CPI": "mean",
        "Unemployment": "mean",
    }
    for col in markdown_cols:
        aggregations[col] = "sum"

    weekly = frame.groupby("Date").agg(aggregations).sort_index().asfreq("W-FRI")
    weekly = weekly.rename(columns={"IsHoliday": "holiday_share"})
    weekly["total_markdown"] = weekly[markdown_cols].sum(axis=1) if markdown_cols else 0.0
    weekly["month"] = weekly.index.month
    weekly["weekofyear"] = weekly.index.isocalendar().week.astype(int)
    weekly["week_sin"] = np.sin(2 * np.pi * weekly["weekofyear"] / 52.0)
    weekly["week_cos"] = np.cos(2 * np.pi * weekly["weekofyear"] / 52.0)
    weekly["is_december"] = (weekly["month"] == 12).astype(int)
    weekly = weekly.replace([np.inf, -np.inf], np.nan).ffill().bfill().fillna(0.0)
    return weekly


def select_exog_features(exog_train, y_train, corr_threshold=CORRELATION_THRESHOLD, max_features=MAX_EXOG_FEATURES):
    candidates = []
    y = pd.Series(y_train, index=exog_train.index).astype(float)
    for col in exog_train.columns:
        values = exog_train[col].astype(float)
        if values.nunique(dropna=False) <= 1:
            continue
        corr = values.corr(y)
        if pd.isna(corr):
            continue
        candidates.append((col, abs(float(corr))))

    ranked = sorted(candidates, key=lambda item: item[1], reverse=True)
    selected = [col for col, corr in ranked if corr >= corr_threshold]
    if not selected:
        selected = [col for col, _ in ranked[:max_features]]

    pruned = []
    corr_matrix = exog_train[selected].corr().abs() if selected else pd.DataFrame()
    for col in selected:
        if len(pruned) >= max_features:
            break
        if all(corr_matrix.loc[col, prev] < 0.95 for prev in pruned):
            pruned.append(col)
    return pruned


weekly_sales_train = weekly_total_sales(train_part)
weekly_exog = build_weekly_exog(features)
exog_train_all = weekly_exog.loc[weekly_sales_train.index]
selected_exog_features = select_exog_features(exog_train_all, weekly_sales_train)

print("Selected SARIMAX exogenous features:")
for feature in selected_exog_features:
    print("-", feature)

Selected SARIMAX exogenous features:
- is_december
- month
- MarkDown3
- total_markdown
- week_cos
- Temperature
- holiday_share
- MarkDown5


## Allocation Strategies

The aggregate SARIMA/SARIMAX model forecasts total weekly sales. Kaggle needs row-level Store-Dept predictions, so we compare two allocation strategies:

- `last_year_share`: uses same Store-Dept sales 52 weeks ago;
- `blended_share`: combines last-year sales with recent pre-validation average sales.

In [9]:
def fit_forecast_sarima(y_train, forecast_dates, order, exog_train=None, exog_future=None):
    model = SARIMAX(
        y_train,
        order=order,
        seasonal_order=(0, 0, 0, 0),
        exog=exog_train,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    result = model.fit(disp=False, maxiter=200)
    forecast = result.get_forecast(steps=len(forecast_dates), exog=exog_future).predicted_mean
    forecast = pd.Series(np.asarray(forecast), index=forecast_dates).clip(lower=0.0)
    return result, forecast


def make_row_forecast(target_frame, history_frame, aggregate_forecast, strategy="last_year_share", recent_weeks=13):
    target = target_frame[["Store", "Dept", "Date", "IsHoliday"]].copy()
    history = history_frame[["Store", "Dept", "Date", "Weekly_Sales"]].copy()

    last_year = history.copy()
    last_year["Date"] = last_year["Date"] + pd.Timedelta(days=364)
    last_year = last_year.rename(columns={"Weekly_Sales": "last_year_sales"})

    series_mean = (
        history.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
        .mean()
        .rename(columns={"Weekly_Sales": "series_mean_sales"})
    )
    recent_cutoff = history["Date"].max() - pd.Timedelta(days=7 * recent_weeks)
    recent_mean = (
        history[history["Date"] > recent_cutoff]
        .groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
        .mean()
        .rename(columns={"Weekly_Sales": "recent_mean_sales"})
    )

    target = target.merge(last_year, on=["Store", "Dept", "Date"], how="left")
    target = target.merge(series_mean, on=["Store", "Dept"], how="left")
    target = target.merge(recent_mean, on=["Store", "Dept"], how="left")

    if strategy == "last_year_share":
        base = target["last_year_sales"].fillna(target["series_mean_sales"])
    elif strategy == "blended_share":
        last_year_base = target["last_year_sales"].fillna(target["series_mean_sales"])
        recent_base = target["recent_mean_sales"].fillna(target["series_mean_sales"])
        base = 0.70 * last_year_base + 0.30 * recent_base
    else:
        raise ValueError(f"Unknown allocation strategy: {strategy}")

    target["allocation_base"] = base.fillna(0.0).clip(lower=0.0)
    date_base_sum = target.groupby("Date")["allocation_base"].transform("sum")
    row_count = target.groupby("Date")["allocation_base"].transform("size")
    target["share"] = np.where(date_base_sum > 0, target["allocation_base"] / date_base_sum, 1.0 / row_count)
    target["aggregate_forecast"] = target["Date"].map(aggregate_forecast)
    target["prediction"] = (target["aggregate_forecast"] * target["share"]).fillna(0.0).clip(lower=0.0)
    return target["prediction"].to_numpy()


def make_seasonal_naive_forecast(target_frame, history_frame):
    target = target_frame[["Store", "Dept", "Date"]].copy()
    history = history_frame[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
    history["Date"] = history["Date"] + pd.Timedelta(days=364)
    history = history.rename(columns={"Weekly_Sales": "last_year_sales"})
    fallback = (
        history_frame.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
        .median()
        .rename(columns={"Weekly_Sales": "series_median_sales"})
    )
    target = target.merge(history, on=["Store", "Dept", "Date"], how="left")
    target = target.merge(fallback, on=["Store", "Dept"], how="left")
    return target["last_year_sales"].fillna(target["series_median_sales"]).fillna(0.0).clip(lower=0.0).to_numpy()

In [10]:
weekly_sales = weekly_total_sales(train_part)
forecast_dates = list(pd.to_datetime(validation_dates))
exog_train_selected = weekly_exog.loc[weekly_sales.index, selected_exog_features]
exog_val_selected = weekly_exog.loc[forecast_dates, selected_exog_features]

seasonal_naive_pred = make_seasonal_naive_forecast(val_part, train_part)
seasonal_naive_wmae = weighted_mae(val_part["Weekly_Sales"], seasonal_naive_pred, val_part["IsHoliday"], HOLIDAY_WEIGHT)
print(f"Seasonal naive WMAE: {seasonal_naive_wmae:.4f}")

wandb_run = None
trial_table = None
if RUN_WANDB:
    if wandb is None:
        raise ImportError("wandb is not available. Re-run the install/import cell or install wandb.")
    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        job_type="sarima-experiment",
        mode=WANDB_MODE,
        reinit=True,
        config={
            "validation_weeks": VALIDATION_WEEKS,
            "holiday_weight": HOLIDAY_WEIGHT,
            "order_grid_size": len(ORDER_GRID),
            "use_exog_options": USE_EXOG_OPTIONS,
            "allocation_strategies": ALLOCATION_STRATEGIES,
            "selected_exog_features": selected_exog_features,
            "seasonal_order": "disabled",
        },
    )
    trial_table = wandb.Table(columns=["trial", "order", "use_exog", "allocation", "wmae", "mae", "rmse", "improvement_vs_seasonal_naive_pct"])

results = []
trial = 0
for order in ORDER_GRID:
    for use_exog in USE_EXOG_OPTIONS:
        exog_train = exog_train_selected if use_exog and selected_exog_features else None
        exog_val = exog_val_selected if use_exog and selected_exog_features else None
        try:
            _, aggregate_forecast = fit_forecast_sarima(weekly_sales, forecast_dates, order, exog_train, exog_val)
        except Exception as exc:
            print(f"Trial {trial:03d} failed for order={order}, use_exog={use_exog}: {exc}")
            trial += 1
            continue

        for allocation in ALLOCATION_STRATEGIES:
            pred = make_row_forecast(val_part, train_part, aggregate_forecast, allocation)
            wmae_value = weighted_mae(val_part["Weekly_Sales"], pred, val_part["IsHoliday"], HOLIDAY_WEIGHT)
            mae_value = float(mean_absolute_error(val_part["Weekly_Sales"], pred))
            rmse_value = rmse(val_part["Weekly_Sales"], pred)
            improvement = 100.0 * (seasonal_naive_wmae - wmae_value) / seasonal_naive_wmae
            row = {
                "trial": trial,
                "order": order,
                "use_exog": use_exog,
                "allocation": allocation,
                "validation/wmae": wmae_value,
                "validation/mae": mae_value,
                "validation/rmse": rmse_value,
                "improvement_vs_seasonal_naive_pct": improvement,
            }
            results.append(row)
            print(f"Trial {trial:03d} | order={order} exog={use_exog} allocation={allocation} | WMAE={wmae_value:.4f} | improvement={improvement:.2f}%")
            if RUN_WANDB:
                wandb.log({
                    "trial": trial,
                    "validation/wmae": wmae_value,
                    "validation/mae": mae_value,
                    "validation/rmse": rmse_value,
                    "improvement_vs_seasonal_naive_pct": improvement,
                    "baseline/seasonal_naive_wmae": seasonal_naive_wmae,
                }, step=trial)
                trial_table.add_data(trial, str(order), use_exog, allocation, wmae_value, mae_value, rmse_value, improvement)
        trial += 1

results_df = pd.DataFrame(results).sort_values("validation/wmae").reset_index(drop=True)
best_result = results_df.iloc[0].to_dict()

if RUN_WANDB:
    wandb.log({"sarima_experiment/trials": trial_table})
    wandb.summary["best_validation_wmae"] = float(best_result["validation/wmae"])
    wandb.summary["best_validation_mae"] = float(best_result["validation/mae"])
    wandb.summary["best_validation_rmse"] = float(best_result["validation/rmse"])
    wandb.summary["best_order"] = str(best_result["order"])
    wandb.summary["best_use_exog"] = bool(best_result["use_exog"])
    wandb.summary["best_allocation"] = best_result["allocation"]
    wandb.summary["seasonal_naive_wmae"] = float(seasonal_naive_wmae)
    wandb.finish()

display(results_df.head(10))
best_result

Seasonal naive WMAE: 1800.1736


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nmetr23 (kende23-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Trial 000 | order=(0, 0, 0) exog=True allocation=last_year_share | WMAE=2563.6915 | improvement=-42.41%
Trial 000 | order=(0, 0, 0) exog=True allocation=blended_share | WMAE=2744.6931 | improvement=-52.47%
Trial 001 | order=(0, 0, 1) exog=True allocation=last_year_share | WMAE=2686.9430 | improvement=-49.26%
Trial 001 | order=(0, 0, 1) exog=True allocation=blended_share | WMAE=2866.8664 | improvement=-59.25%
Trial 002 | order=(0, 0, 2) exog=True allocation=last_year_share | WMAE=2768.5633 | improvement=-53.79%
Trial 002 | order=(0, 0, 2) exog=True allocation=blended_share | WMAE=2947.3829 | improvement=-63.73%
Trial 003 | order=(0, 1, 0) exog=True allocation=last_year_share | WMAE=3840.8970 | improvement=-113.36%
Trial 003 | order=(0, 1, 0) exog=True allocation=blended_share | WMAE=3985.6954 | improvement=-121.41%
Trial 004 | order=(0, 1, 1) exog=True allocation=last_year_share | WMAE=3545.5354 | improvement=-96.96%
Trial 004 | order=(0, 1, 1) exog=True allocation=blended_share | WMAE=

baseline/seasonal_naive_wmae,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
improvement_vs_seasonal_naive_pct,█▇▇▃▄▃▇▇▆▄▄▁▆▆▇▄▄▂
trial,▁▁▂▂▃▃▃▄▄▅▅▆▆▆▇▇██
validation/mae,▁▂▂▆▅▆▂▂▃▅▅█▂▂▂▅▅▇
validation/rmse,▁▂▂▆▅▆▂▂▃▅▅█▃▃▂▅▅▇
validation/wmae,▁▂▂▆▅▆▂▂▃▅▅█▃▃▂▅▅▇
baseline/seasonal_naive_wmae,1800.17359
best_allocation,last_year_share
best_order,"(0, 0, 0)"
best_use_exog,True
best_validation_mae,2517.84864


,trial,order,use_exog,allocation,validation/wmae,validation/mae,validation/rmse,improvement_vs_seasonal_naive_pct
0,0,"(0, 0, 0)",True,last_year_share,2563.691454,2517.848644,5135.390683,-42.413569
1,1,"(0, 0, 1)",True,last_year_share,2686.943027,2639.074386,5403.205692,-49.260218
2,14,"(2, 0, 2)",True,last_year_share,2691.230807,2645.040971,5457.209562,-49.498405
3,0,"(0, 0, 0)",True,blended_share,2744.693098,2698.474898,5397.509325,-52.468246
4,2,"(0, 0, 2)",True,last_year_share,2768.563271,2692.662390,5525.380359,-53.794239
5,6,"(1, 0, 0)",True,last_year_share,2793.698798,2728.395462,5607.698710,-55.190522
6,1,"(0, 0, 1)",True,blended_share,2866.866364,2818.122614,5667.633265,-59.254995
7,14,"(2, 0, 2)",True,blended_share,2868.296649,2822.275758,5716.041538,-59.334448
8,7,"(1, 0, 1)",True,last_year_share,2900.339719,2817.951074,5743.310563,-61.114447
9,12,"(2, 0, 0)",True,last_year_share,2936.653875,2829.923080,5784.496908,-63.131705


{'trial': 0,
 'order': (0, 0, 0),
 'use_exog': True,
 'allocation': 'last_year_share',
 'validation/wmae': 2563.69145432203,
 'validation/mae': 2517.8486443252004,
 'validation/rmse': 5135.390682821048,
 'improvement_vs_seasonal_naive_pct': -42.413568679397436}

:## Register Best SARIMAX Pipeline

Run this cell after the experiment loop. It refits the best SARIMAX order on all available training data, packages the fitted aggregate model with selected exogenous features and allocation history, logs it as a W&B model artifact, and links it to the W&B Model Registry.


In [11]:
required_objects = ["best_result", "train", "features", "weekly_total_sales", "build_weekly_exog", "selected_exog_features"]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError("Run previous experiment cells first. Missing: " + ", ".join(missing_objects))
if wandb is None:
    raise ImportError("wandb is not available. Re-run the install/import cell or install wandb.")

REGISTER_SARIMAX_MODEL = True
SARIMAX_REGISTRY_TARGET = "wandb-registry-model/Walmart_SARIMAX_Pipeline"
SARIMAX_MODEL_ARTIFACT_NAME = "walmart-sarimax-best-pipeline"


class AggregateSARIMAXPipeline:
    """Raw-test SARIMAX pipeline for W&B Model Registry."""

    def __init__(self, model_result, order, allocation_strategy, observed_history, feature_history, selected_features, validation_metrics, metadata):
        self.model_result = model_result
        self.order = tuple(order)
        self.allocation_strategy = allocation_strategy
        self.observed_history = observed_history.copy()
        self.feature_history = feature_history.copy()
        self.selected_features = list(selected_features)
        self.validation_metrics = dict(validation_metrics)
        self.metadata = dict(metadata)

    def _build_weekly_exog(self, features_frame):
        frame = features_frame.copy()
        markdown_cols = [c for c in frame.columns if c.startswith("MarkDown")]
        for col in markdown_cols:
            frame[col] = frame[col].fillna(0.0)
        numeric_cols = ["Temperature", "Fuel_Price", "CPI", "Unemployment"] + markdown_cols
        for col in numeric_cols:
            if col in frame.columns:
                frame[col] = frame[col].fillna(frame[col].median())
        aggregations = {
            "IsHoliday": "mean",
            "Temperature": "mean",
            "Fuel_Price": "mean",
            "CPI": "mean",
            "Unemployment": "mean",
        }
        for col in markdown_cols:
            aggregations[col] = "sum"
        weekly = frame.groupby("Date").agg(aggregations).sort_index().asfreq("W-FRI")
        weekly = weekly.rename(columns={"IsHoliday": "holiday_share"})
        weekly["total_markdown"] = weekly[markdown_cols].sum(axis=1) if markdown_cols else 0.0
        weekly["month"] = weekly.index.month
        weekly["weekofyear"] = weekly.index.isocalendar().week.astype(int)
        weekly["week_sin"] = np.sin(2 * np.pi * weekly["weekofyear"] / 52.0)
        weekly["week_cos"] = np.cos(2 * np.pi * weekly["weekofyear"] / 52.0)
        weekly["is_december"] = (weekly["month"] == 12).astype(int)
        return weekly.replace([np.inf, -np.inf], np.nan).ffill().bfill().fillna(0.0)

    def _aggregate_forecast(self, raw_df, features_frame=None):
        dates = list(pd.to_datetime(sorted(raw_df["Date"].unique())))
        source_features = self.feature_history if features_frame is None else features_frame.copy()
        source_features["Date"] = pd.to_datetime(source_features["Date"])
        weekly_exog_future = self._build_weekly_exog(source_features).loc[dates, self.selected_features]
        forecast = self.model_result.get_forecast(steps=len(dates), exog=weekly_exog_future).predicted_mean
        return pd.Series(np.asarray(forecast), index=dates).clip(lower=0.0)

    def predict(self, raw_df, features_frame=None):
        frame = raw_df.copy()
        frame["Date"] = pd.to_datetime(frame["Date"])
        if "IsHoliday" not in frame.columns:
            frame["IsHoliday"] = False
        aggregate_forecast = self._aggregate_forecast(frame, features_frame=features_frame)
        return self._make_row_forecast(frame, aggregate_forecast)

    def _make_row_forecast(self, target_frame, aggregate_forecast, recent_weeks=13):
        target = target_frame[["Store", "Dept", "Date", "IsHoliday"]].copy()
        history = self.observed_history[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
        history["Date"] = pd.to_datetime(history["Date"])
        last_year = history.copy()
        last_year["Date"] = last_year["Date"] + pd.Timedelta(days=364)
        last_year = last_year.rename(columns={"Weekly_Sales": "last_year_sales"})
        series_mean = (
            history.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
            .mean()
            .rename(columns={"Weekly_Sales": "series_mean_sales"})
        )
        recent_cutoff = history["Date"].max() - pd.Timedelta(days=7 * recent_weeks)
        recent_mean = (
            history[history["Date"] > recent_cutoff]
            .groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
            .mean()
            .rename(columns={"Weekly_Sales": "recent_mean_sales"})
        )
        target = target.merge(last_year, on=["Store", "Dept", "Date"], how="left")
        target = target.merge(series_mean, on=["Store", "Dept"], how="left")
        target = target.merge(recent_mean, on=["Store", "Dept"], how="left")
        if self.allocation_strategy == "last_year_share":
            base = target["last_year_sales"].fillna(target["series_mean_sales"])
        elif self.allocation_strategy == "blended_share":
            last_year_base = target["last_year_sales"].fillna(target["series_mean_sales"])
            recent_base = target["recent_mean_sales"].fillna(target["series_mean_sales"])
            base = 0.70 * last_year_base + 0.30 * recent_base
        else:
            raise ValueError(f"Unknown allocation strategy: {self.allocation_strategy}")
        target["allocation_base"] = base.fillna(0.0).clip(lower=0.0)
        date_base_sum = target.groupby("Date")["allocation_base"].transform("sum")
        row_count = target.groupby("Date")["allocation_base"].transform("size")
        target["share"] = np.where(date_base_sum > 0, target["allocation_base"] / date_base_sum, 1.0 / row_count)
        target["aggregate_forecast"] = target["Date"].map(aggregate_forecast)
        target["prediction"] = (target["aggregate_forecast"] * target["share"]).fillna(0.0).clip(lower=0.0)
        return target["prediction"].to_numpy()


if REGISTER_SARIMAX_MODEL:
    best_order = tuple(best_result["order"])
    best_allocation = best_result["allocation"]
    full_weekly_sales = weekly_total_sales(train)
    full_weekly_exog = build_weekly_exog(features)
    full_exog_train = full_weekly_exog.loc[full_weekly_sales.index, selected_exog_features]
    full_model = SARIMAX(
        full_weekly_sales,
        order=best_order,
        seasonal_order=(0, 0, 0, 0),
        exog=full_exog_train,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False, maxiter=200)

    registry_output_dir = OUTPUT_DIR / "sarimax_registry"
    registry_output_dir.mkdir(parents=True, exist_ok=True)
    pipeline_path = registry_output_dir / "sarimax_best_pipeline.pkl"

    validation_metrics = {
        "best_validation_wmae": float(best_result["validation/wmae"]),
        "best_validation_mae": float(best_result["validation/mae"]),
        "best_validation_rmse": float(best_result["validation/rmse"]),
        "seasonal_naive_wmae": float(seasonal_naive_wmae),
        "improvement_vs_seasonal_naive_pct": float(best_result["improvement_vs_seasonal_naive_pct"]),
    }
    pipeline = AggregateSARIMAXPipeline(
        model_result=full_model,
        order=best_order,
        allocation_strategy=best_allocation,
        observed_history=train,
        feature_history=features,
        selected_features=selected_exog_features,
        validation_metrics=validation_metrics,
        metadata={
            "model_family": "SARIMAX",
            "uses_exog": True,
            "selected_exog_features": selected_exog_features,
            "seasonal_order": "disabled",
            "validation_weeks": VALIDATION_WEEKS,
            "refit_scope": "full_train",
        },
    )
    with pipeline_path.open("wb") as file:
        cloudpickle.dump(pipeline, file)

    registry_run = wandb.init(
        project=WANDB_PROJECT,
        name="SARIMAX_Best_Model_Registry",
        job_type="model_registration",
        mode=WANDB_MODE,
        reinit=True,
        config={
            **validation_metrics,
            "best_order": str(best_order),
            "best_allocation": best_allocation,
            "selected_exog_features": selected_exog_features,
            "registry_target": SARIMAX_REGISTRY_TARGET,
        },
    )
    model_artifact = wandb.Artifact(
        name=SARIMAX_MODEL_ARTIFACT_NAME,
        type="model",
        description="Best aggregate SARIMAX pipeline with selected exogenous features and row-level Store-Dept allocation.",
        metadata={
            **validation_metrics,
            "model_family": "SARIMAX",
            "order": str(best_order),
            "allocation_strategy": best_allocation,
            "selected_exog_features": selected_exog_features,
            "registry_target": SARIMAX_REGISTRY_TARGET,
        },
    )
    model_artifact.add_file(str(pipeline_path))
    logged_artifact = registry_run.log_artifact(model_artifact, aliases=["best", "latest"])
    registry_run.link_artifact(logged_artifact, target_path=SARIMAX_REGISTRY_TARGET, aliases=["best-sarimax", "latest", "champion"])
    registry_run.summary.update(validation_metrics)
    registry_run.summary["registry_target"] = SARIMAX_REGISTRY_TARGET
    registry_run.summary["pipeline_artifact"] = SARIMAX_MODEL_ARTIFACT_NAME
    registry_run.finish()

    print("Registered best SARIMAX pipeline in W&B Model Registry:")
    print(f"  artifact: {SARIMAX_MODEL_ARTIFACT_NAME}")
    print(f"  registry: {SARIMAX_REGISTRY_TARGET}")
    print(f"  validation WMAE: {validation_metrics['best_validation_wmae']:.4f}")


best_validation_mae,2517.84864
best_validation_rmse,5135.39068
best_validation_wmae,2563.69145
improvement_vs_seasonal_naive_pct,-42.41357
pipeline_artifact,walmart-sarimax-best...
registry_target,wandb-registry-model...
seasonal_naive_wmae,1800.17359


Registered best SARIMAX pipeline in W&B Model Registry:
  artifact: walmart-sarimax-best-pipeline
  registry: wandb-registry-model/Walmart_SARIMAX_Pipeline
  validation WMAE: 2563.6915
